## VALUE INVESTING USING SOME IMP RATIOS 

In [15]:
import pandas as pd
import numpy as np
import yfinance as yf
import math
from scipy import stats

In [5]:
tickers = pd.read_csv("nse_ticker_name_list.csv")
tickers.head()

,Ticker,Ticker.1,Company Name
0,20MICRONS,20MICRONS.NS,20 Microns Limited
1,21STCENMGM,21STCENMGM.NS,21st Century Management Services Limited
2,360ONE,360ONE.NS,360 ONE WAM LIMITED
3,3BBLACKBIO,3BBLACKBIO.NS,3B Blackbio Dx Limited
4,3IINFOLTD,3IINFOLTD.NS,3i Infotech Limited


In [9]:
def fetch_values_of_stocks(tickers):

    value_columns = [
        "Ticker",
        "Price",
        "PE-Ratio",
        "PB-Ratio",
        "PS-Ratio",
        "PEG-Ratio",
        "EV/EBITDA",
        "EPS",
        "DEBT_TO_EQUITY",
        "ROE"
    ]
    value_df = pd.DataFrame(columns=value_columns)
    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period="1d")
            if hist.empty:
                continue
            price = hist["Close"].iloc[-1]
            

            financials = stock.financials # later go and check stock. there are many options
            balanceSheet = stock.balance_sheet
            cashFlow = stock.cashflow

            try:                                      
                info = stock.info                     
            except TypeError:                        
                info = {}                            


            Price_earning = stock.info.get("trailingPE", np.nan)
            price_book = stock.info.get("priceToBook", np.nan)
            price_to_sales = stock.info.get("priceToSalesTrailing12Months", np.nan)
            peg_ratio = stock.info.get("pegRatio", np.nan)
            ev = stock.info.get("enterpriseValue", np.nan)
            ebitda = stock.info.get("ebitda", np.nan)
            evEbidta = ev / ebitda if ev and ebitda else np.nan
            eps = stock.info.get("trailingEps", np.nan)
            Debt_to_equity = stock.info.get("debtToEquity", np.nan)
            roe = stock.info.get("returnOnEquity", np.nan)

            value_df.loc[len(value_df)] = [
                ticker,
                price,
                Price_earning,
                price_book,
                price_to_sales,
                peg_ratio,
                evEbidta,
                eps,
                Debt_to_equity,
                roe 
            ]

        except Exception as e:                       
            print(f"Skipping {ticker}: {e}") 
    
    return value_df

        


# Checking all the ratio and data info of a stock

In [10]:
ril = yf.Ticker("ITC.NS")
ril.info

{'address1': 'Virginia House',
 'address2': '37 Jawaharlal Nehru Road',
 'city': 'Kolkata',
 'zip': '700071',
 'country': 'India',
 'phone': '91 33 2288 9371',
 'fax': '91 33 2288 0655',
 'website': 'https://www.itcportal.com',
 'industry': 'Tobacco',
 'industryKey': 'tobacco',
 'industryDisp': 'Tobacco',
 'sector': 'Consumer Defensive',
 'sectorKey': 'consumer-defensive',
 'sectorDisp': 'Consumer Defensive',
 'longBusinessSummary': 'ITC Limited engages in the fast-moving consumer goods, paperboards, paper and packaging, and agri businesses in India and internationally. The company offers cigarettes and cigars; foods, including staples, spices, biscuits, confectionery and gums, snacks, noodles and pasta, beverages, dairy products, ready-to-eat meals, chocolates, coffee, and frozen foods; personal care products; education and stationery, such as balls, gels and roller pens, mechanical pencils, geometry boxes, erasers, sharpeners and rulers, wax crayons, plastic crayons, and sketch pens 

In [11]:
tickers_list = tickers["Ticker.1"].values.tolist()
df = fetch_values_of_stocks(tickers_list)
df.head(100)

$KALYANI.NS: possibly delisted; no price data found  (period=1d)
$MIRCELECTR.NS: possibly delisted; no price data found  (period=1d)
$RELTD-RE.NS: possibly delisted; no price data found  (period=1d)
$SHAH-RE1.NS: possibly delisted; no price data found  (period=1d)
$SICALLOG.NS: possibly delisted; no price data found  (period=1d)
Failed to perform, curl: (16) . See https://curl.se/libcurl/c/libcurl-errors.html first for more details.


Skipping SPORTKING.NS: 'NoneType' object has no attribute 'get'


,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,PEG-Ratio,EV/EBITDA,EPS,DEBT_TO_EQUITY,ROE
0,20MICRONS.NS,206.410004,10.898099,1.506807,0.763602,NaN,6.032545,18.94,32.043,0.14479
1,21STCENMGM.NS,33.209999,NaN,1.054620,-2.066045,NaN,-1.335999,-22.90,NaN,-0.57208
2,360ONE.NS,1116.900024,38.30247,4.605146,10.145189,NaN,21.488595,29.16,161.977,0.14392
3,3BBLACKBIO.NS,1181.099976,NaN,3.041156,NaN,NaN,16.689503,NaN,0.937,0.19889
4,3IINFOLTD.NS,17.389999,9.554944,0.975487,0.520184,NaN,-19.578284,1.82,16.298,0.10256
...,...,...,...,...,...,...,...,...,...,...
95,AKG.NS,11.130000,101.181816,0.720902,0.406495,NaN,190.982512,0.11,10.820,0.00683
96,AKI.NS,4.820000,25.368422,0.624919,0.463490,NaN,11.859639,0.19,21.381,0.02909
97,AKSHAR.NS,0.450000,NaN,0.433526,0.302146,NaN,-46.044088,-0.09,6.226,-0.08741
98,AKSHARCHEM.NS,238.080002,NaN,0.731143,0.513525,NaN,17.518279,-0.54,41.473,-0.00167


In [14]:
value_columns = [
    
        "PE-Ratio",
        "PB-Ratio",
        "PS-Ratio",
        "PEG-Ratio",
        "EV/EBITDA",
        "EPS",
        "DEBT_TO_EQUITY",
        "ROE"
    ]
numeric_cols = df.columns.drop("Ticker")  # exclude the text column
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].mean())   

df.info()

<class 'pandas.DataFrame'>
Index: 2368 entries, 0 to 2367
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Ticker          2368 non-null   str    
 1   Price           2368 non-null   float64
 2   PE-Ratio        2368 non-null   float64
 3   PB-Ratio        2368 non-null   float64
 4   PS-Ratio        2368 non-null   float64
 5   PEG-Ratio       2368 non-null   float64
 6   EV/EBITDA       2368 non-null   float64
 7   EPS             2368 non-null   float64
 8   DEBT_TO_EQUITY  2368 non-null   float64
 9   ROE             2368 non-null   float64
dtypes: float64(9), str(1)
memory usage: 203.5 KB


## Taking out the Percentile for each column of the Stocks

In [19]:
positive_only = ["PE-Ratio", "PB-Ratio", "PS-Ratio", "PEG-Ratio", "EV/EBITDA", "EPS", "DEBT_TO_EQUITY", "ROE"]
df[positive_only] = df[positive_only].replace([np.inf, -np.inf], np.nan).clip(lower=0).replace(0, np.nan)

percentile_metrices = {
    "PE-Ratio"       : "PE-Ratio_Percntile",
    "PB-Ratio"       : "PB-Ratio_Percntile",
    "PS-Ratio"       : "PS-Ratio_Percntile",
    "PEG-Ratio"      : "PEG-Ratio_Percntile",
    "EV/EBITDA"      : "EV/EBITDA_Percentile",
    "EPS"            : "EPS_Percentile",
    "DEBT_TO_EQUITY" : "DEBT_TO_EQUITY_Percentile",
    "ROE"            : "ROE_Percentile"
}

for metric, percentile in percentile_metrices.items():
    valid = df[metric].dropna()
    df[percentile] = df[metric].apply(
        lambda x: stats.percentileofscore(valid, x) / 100 if pd.notna(x) else np.nan
    )

df.head()

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,PEG-Ratio,EV/EBITDA,EPS,DEBT_TO_EQUITY,ROE,PE-Ratio_Percntile,PB-Ratio_Percntile,PS-Ratio_Percntile,PEG-Ratio_Percntile,EV/EBITDA_Percentile,EPS_Percentile,DEBT_TO_EQUITY_Percentile,ROE_Percentile
0,20MICRONS.NS,206.410004,10.898099,1.506807,0.763602,2.173068,6.032545,18.940000,32.043000,0.14479,0.120980,0.318821,0.183362,0.513725,0.053315,0.619980,0.466413,0.692560
1,21STCENMGM.NS,33.209999,NaN,1.054620,NaN,2.173068,NaN,NaN,66.969787,NaN,NaN,0.203166,NaN,0.513725,NaN,NaN,0.713139,NaN
2,360ONE.NS,1116.900024,38.302470,4.605146,10.145189,2.173068,21.488595,29.160000,161.977000,0.14392,0.647269,0.735268,0.850170,0.513725,0.559573,0.757591,0.925222,0.687880
3,3BBLACKBIO.NS,1181.099976,NaN,3.041156,62.201757,2.173068,16.689503,26.507703,0.937000,0.19889,NaN,0.579595,0.958192,0.513725,0.451553,0.719393,0.064216,0.836687
4,3IINFOLTD.NS,17.389999,9.554944,0.975487,0.520184,2.173068,NaN,1.820000,16.298000,0.10256,0.087800,0.186456,0.109932,0.513725,NaN,0.170911,0.323194,0.503042


In [ ]:

df["Value Score"] = df[[value for value in percentile_metrices.values()]].mean(axis= 1)

df

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,PEG-Ratio,EV/EBITDA,EPS,DEBT_TO_EQUITY,ROE,PE-Ratio_Percntile,PB-Ratio_Percntile,PS-Ratio_Percntile,PEG-Ratio_Percntile,EV/EBITDA_Percentile,EPS_Percentile,DEBT_TO_EQUITY_Percentile,ROE_Percentile,Value Score
0,20MICRONS.NS,206.410004,10.898099,1.506807,0.763602,2.173068,6.032545,18.940000,32.043000,0.144790,0.120980,0.318821,0.183362,0.513725,0.053315,0.619980,0.466413,0.692560,0.371144
1,21STCENMGM.NS,33.209999,NaN,1.054620,NaN,2.173068,NaN,NaN,66.969787,NaN,NaN,0.203166,NaN,0.513725,NaN,NaN,0.713139,NaN,0.476677
2,360ONE.NS,1116.900024,38.302470,4.605146,10.145189,2.173068,21.488595,29.160000,161.977000,0.143920,0.647269,0.735268,0.850170,0.513725,0.559573,0.757591,0.925222,0.687880,0.709587
3,3BBLACKBIO.NS,1181.099976,NaN,3.041156,62.201757,2.173068,16.689503,26.507703,0.937000,0.198890,NaN,0.579595,0.958192,0.513725,0.451553,0.719393,0.064216,0.836687,0.589052
4,3IINFOLTD.NS,17.389999,9.554944,0.975487,0.520184,2.173068,NaN,1.820000,16.298000,0.102560,0.087800,0.186456,0.109932,0.513725,NaN,0.170911,0.323194,0.503042,0.270723
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2363,ZSARACOM.NS,9400.000000,NaN,4.239780,62.201757,2.173068,8.966042,26.507703,0.011000,0.088760,NaN,0.702507,0.958192,0.513725,0.167362,0.719393,0.002957,0.438465,0.500371
2364,ZUARI.NS,235.350006,1.076083,0.391373,0.249683,2.173068,4.107512,218.710000,30.705000,0.077344,0.004084,0.036500,0.034805,0.513725,0.017617,0.984329,0.457119,0.334815,0.297874
2365,ZUARIIND.NS,264.149994,7.284887,0.217288,0.752921,2.173068,41.216474,36.260000,72.104000,0.024500,0.046452,0.012313,0.181239,0.513725,0.867872,0.802155,0.791297,0.089378,0.413054
2366,ZYDUSLIFE.NS,1112.199951,22.226217,4.422407,4.086516,2.173068,16.486691,50.040000,42.242000,0.183180,0.395100,0.722955,0.654924,0.513725,0.444599,0.858472,0.537812,0.801591,0.616147


## Now we will Sort the best Value score Stocks

In [24]:
df = df.sort_values(by= "Value Score", ascending= False)
metric_cols = ["PE-Ratio", "PB-Ratio", "PS-Ratio", "PEG-Ratio", "EV/EBITDA", "EPS", "DEBT_TO_EQUITY", "ROE"]
df = df.dropna(subset=metric_cols, thresh=6)
df

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,PEG-Ratio,EV/EBITDA,EPS,DEBT_TO_EQUITY,ROE,PE-Ratio_Percntile,PB-Ratio_Percntile,PS-Ratio_Percntile,PEG-Ratio_Percntile,EV/EBITDA_Percentile,EPS_Percentile,DEBT_TO_EQUITY_Percentile,ROE_Percentile,Value Score
367,BSE.NS,3941.800049,65.164490,20.360958,31.241250,2.173068,38.253489,60.49,NaN,0.36003,0.826442,0.974494,0.922750,0.513725,0.850719,0.891038,NaN,0.965372,0.849220
2143,TITAN.NS,4305.299805,75.162360,24.340918,4.360381,2.173068,48.914428,57.28,195.001,0.37128,0.860643,0.982410,0.674024,0.513725,0.899397,0.881244,0.939163,0.966776,0.839673
1436,NETWEB.NS,4925.100098,135.715070,38.671307,12.843164,2.173068,98.264149,36.29,39.019,0.32835,0.936192,0.991645,0.879457,0.513725,0.964766,0.802889,0.518800,0.958353,0.820728
2152,TORNTPHARM.NS,4514.100098,70.820520,18.094978,10.928513,2.173068,38.838854,63.74,85.446,0.16986,0.850434,0.968338,0.860357,0.513725,0.854427,0.900588,0.833545,0.770707,0.819015
1956,SOLARINDS.NS,18227.000000,97.863090,26.274668,16.765661,2.173068,63.369594,186.25,23.271,0.31332,0.899438,0.984609,0.894312,0.513725,0.931850,0.979922,0.385298,0.948994,0.817268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,ANIKINDS.NS,44.910000,NaN,0.391383,0.395049,2.173068,3.594662,NaN,3.695,0.07674,NaN,0.036939,0.073005,0.513725,0.010199,NaN,0.146599,0.273280,0.175625
1555,PATELENG.NS,32.119999,11.309859,0.751415,0.601170,1.260000,5.853998,2.84,27.377,0.06550,0.128127,0.126649,0.137097,0.027238,0.049606,0.219148,0.426700,0.228825,0.167924
1423,NDL.NS,2.520000,10.956521,0.557892,0.126486,2.173068,5.083126,0.23,18.050,0.05217,0.123022,0.072999,0.010611,0.513725,0.030598,0.043585,0.343473,0.181095,0.164888
1213,LIBAS.NS,11.750000,3.405797,0.378849,0.396449,2.173068,4.317259,3.45,16.922,0.00248,0.011741,0.032102,0.074278,0.513725,0.020399,0.240206,0.329954,0.009359,0.153970


In [ ]:
df = df.head(100)  # it will take only 10 stocks which got sorted.
df = df.reset_index(drop= True) # resets the index 
df

,level_0,index,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,PEG-Ratio,EV/EBITDA,EPS,...,ROE,PE-Ratio_Percntile,PB-Ratio_Percntile,PS-Ratio_Percntile,PEG-Ratio_Percntile,EV/EBITDA_Percentile,EPS_Percentile,DEBT_TO_EQUITY_Percentile,ROE_Percentile,Value Score
0,0,367,BSE.NS,3941.800049,65.164490,20.360958,31.241250,2.173068,38.253489,60.490000,...,0.36003,0.826442,0.974494,0.922750,0.513725,0.850719,0.891038,NaN,0.965372,0.849220
1,1,2143,TITAN.NS,4305.299805,75.162360,24.340918,4.360381,2.173068,48.914428,57.280000,...,0.37128,0.860643,0.982410,0.674024,0.513725,0.899397,0.881244,0.939163,0.966776,0.839673
2,2,1436,NETWEB.NS,4925.100098,135.715070,38.671307,12.843164,2.173068,98.264149,36.290000,...,0.32835,0.936192,0.991645,0.879457,0.513725,0.964766,0.802889,0.518800,0.958353,0.820728
3,3,2152,TORNTPHARM.NS,4514.100098,70.820520,18.094978,10.928513,2.173068,38.838854,63.740000,...,0.16986,0.850434,0.968338,0.860357,0.513725,0.854427,0.900588,0.833545,0.770707,0.819015
4,4,1956,SOLARINDS.NS,18227.000000,97.863090,26.274668,16.765661,2.173068,63.369594,186.250000,...,0.31332,0.899438,0.984609,0.894312,0.513725,0.931850,0.979922,0.385298,0.948994,0.817268
5,5,1922,SIKA.NS,1116.349976,NaN,15.220118,62.201757,2.173068,51.827585,26.507703,...,0.26080,NaN,0.957344,0.958192,0.513725,0.906351,0.719393,0.713139,0.912962,0.811587
6,6,151,APOLLOHOSP.NS,8487.500000,63.029110,12.870184,4.837277,2.173068,34.117408,134.660000,...,0.21503,0.812660,0.940633,0.700764,0.513725,0.825220,0.967679,0.832700,0.867571,0.807619
7,7,1842,SCHNEIDER.NS,1442.199951,161.500550,52.344654,11.929435,2.173068,93.397242,8.930000,...,0.31813,0.947933,0.996042,0.873939,0.513725,0.963839,0.406954,0.784537,0.953673,0.805080
8,8,1919,SIGMAADV.NS,578.849976,38.032192,21.774374,20.533960,2.173068,198.926189,15.220000,...,0.87583,0.644206,0.978012,0.903650,0.513725,0.984701,0.559011,0.785171,0.995788,0.795533
9,9,2166,TRENT.NS,3142.899902,97.848694,23.993800,8.348502,2.173068,61.227727,32.120000,...,0.27126,0.898928,0.981970,0.820883,0.513725,0.928605,0.781097,0.493874,0.920449,0.792441


In [39]:
portfolio_size = int(input("Enter the Amount you want to Invest: "))
# to divide the money we will divide the money by the len of df here 10 as we taking top 10 stocks only. 
position_size = portfolio_size / len(df.index) # it will give how much i will invest in each stock
position_size

100000.0

In [40]:
df["Number Of Shares to buy"] = df['Price'].apply(lambda price: math.floor(
    position_size / price
)) # could have written lambda price : position_size // price.

df

,level_0,index,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,PEG-Ratio,EV/EBITDA,EPS,...,PE-Ratio_Percntile,PB-Ratio_Percntile,PS-Ratio_Percntile,PEG-Ratio_Percntile,EV/EBITDA_Percentile,EPS_Percentile,DEBT_TO_EQUITY_Percentile,ROE_Percentile,Value Score,Number Of Shares to buy
0,0,367,BSE.NS,3941.800049,65.164490,20.360958,31.241250,2.173068,38.253489,60.490000,...,0.826442,0.974494,0.922750,0.513725,0.850719,0.891038,NaN,0.965372,0.849220,25
1,1,2143,TITAN.NS,4305.299805,75.162360,24.340918,4.360381,2.173068,48.914428,57.280000,...,0.860643,0.982410,0.674024,0.513725,0.899397,0.881244,0.939163,0.966776,0.839673,23
2,2,1436,NETWEB.NS,4925.100098,135.715070,38.671307,12.843164,2.173068,98.264149,36.290000,...,0.936192,0.991645,0.879457,0.513725,0.964766,0.802889,0.518800,0.958353,0.820728,20
3,3,2152,TORNTPHARM.NS,4514.100098,70.820520,18.094978,10.928513,2.173068,38.838854,63.740000,...,0.850434,0.968338,0.860357,0.513725,0.854427,0.900588,0.833545,0.770707,0.819015,22
4,4,1956,SOLARINDS.NS,18227.000000,97.863090,26.274668,16.765661,2.173068,63.369594,186.250000,...,0.899438,0.984609,0.894312,0.513725,0.931850,0.979922,0.385298,0.948994,0.817268,5
5,5,1922,SIKA.NS,1116.349976,NaN,15.220118,62.201757,2.173068,51.827585,26.507703,...,NaN,0.957344,0.958192,0.513725,0.906351,0.719393,0.713139,0.912962,0.811587,89
6,6,151,APOLLOHOSP.NS,8487.500000,63.029110,12.870184,4.837277,2.173068,34.117408,134.660000,...,0.812660,0.940633,0.700764,0.513725,0.825220,0.967679,0.832700,0.867571,0.807619,11
7,7,1842,SCHNEIDER.NS,1442.199951,161.500550,52.344654,11.929435,2.173068,93.397242,8.930000,...,0.947933,0.996042,0.873939,0.513725,0.963839,0.406954,0.784537,0.953673,0.805080,69
8,8,1919,SIGMAADV.NS,578.849976,38.032192,21.774374,20.533960,2.173068,198.926189,15.220000,...,0.644206,0.978012,0.903650,0.513725,0.984701,0.559011,0.785171,0.995788,0.795533,172
9,9,2166,TRENT.NS,3142.899902,97.848694,23.993800,8.348502,2.173068,61.227727,32.120000,...,0.898928,0.981970,0.820883,0.513725,0.928605,0.781097,0.493874,0.920449,0.792441,31
